# Flightpath: U.S. airline reliability

24,416,952 scheduled flights, January 2023–June 2026. BTS reporting-carrier records.

This notebook reads the same corrected model used by Excel and Tableau. Arrival rates exclude cancelled, diverted and missing-arrival records. Half-year comparisons use January–June in every year. Patterns are descriptive; a matched comparison does not identify a causal effect.

In [1]:
from pathlib import Path
import duckdb
root=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
con=duckdb.connect(str(root/'data/processed/flightpath.duckdb'),read_only=True)

## Comparable January–June windows
The partial 2026 release is compared with the same six months in earlier years.

In [2]:
con.execute('SELECT year,sum(scheduled_flights) AS scheduled_flights,sum(eligible_arrivals) AS eligible_arrivals,sum(on_time_flights)*1.0/sum(eligible_arrivals) AS on_time_rate,sum(cancelled_flights)*1.0/sum(scheduled_flights) AS cancellation_rate FROM monthly WHERE month<=6 GROUP BY 1 ORDER BY 1').fetchdf()

,year,scheduled_flights,eligible_arrivals,on_time_rate,cancellation_rate
0,2023,3340569.0,3279959.0,0.778048,0.015518
1,2024,3461319.0,3403465.0,0.780034,0.014060
2,2025,3446676.0,3385822.0,0.781054,0.014981
3,2026,3488373.0,3402621.0,0.778782,0.021810


## Complete years
2026 is intentionally absent from a full-year comparison.

In [3]:
con.execute('SELECT year,sum(scheduled_flights) AS scheduled_flights,sum(eligible_arrivals) AS eligible_arrivals,sum(on_time_flights)*1.0/sum(eligible_arrivals) AS on_time_rate FROM monthly WHERE year<=2025 GROUP BY 1 ORDER BY 1').fetchdf()

,year,scheduled_flights,eligible_arrivals,on_time_rate
0,2023,6847899.0,6743403.0,0.794362
1,2024,7079061.0,6965247.0,0.791828
2,2025,7001619.0,6879484.0,0.776925


## Morning versus evening within comparable operating cells
Cells use the same directional route, reporting carrier and calendar month. At least 30 eligible arrivals are required in each period. Common weights equal the smaller of the morning/evening counts. Coverage is reported; results do not generalize automatically to unmatched cells.

In [4]:
con.execute('SELECT year,sum(common_weight*morning_rate)/sum(common_weight) AS morning_rate,sum(common_weight*evening_rate)/sum(common_weight) AS evening_rate,sum(common_weight*gap)/sum(common_weight) AS gap,count(*) AS matched_cells,sum(morning_n+evening_n) AS represented_arrivals FROM matched_dayparts WHERE month<=6 GROUP BY 1 ORDER BY 1').fetchdf()

,year,morning_rate,evening_rate,gap,matched_cells,represented_arrivals
0,2023,0.143986,0.311968,0.167981,4134,430242.0
1,2024,0.144144,0.317382,0.173238,4141,425578.0
2,2025,0.131519,0.290381,0.158863,4311,440108.0
3,2026,0.137376,0.297063,0.159686,4097,424196.0


## Sensitivity to sample-size thresholds
Check whether the pattern survives stricter minimum counts. All versions use the same common-weight formula.

In [5]:
con.execute('SELECT threshold,year,count(*) AS matched_cells,sum(common_weight*gap)/sum(common_weight) AS gap FROM matched_dayparts CROSS JOIN (VALUES (30),(60),(100)) limits(threshold) WHERE month<=6 AND morning_n>=threshold AND evening_n>=threshold GROUP BY 1,2 ORDER BY 1,2').fetchdf()

,threshold,year,matched_cells,gap
0,30,2023,4134,0.167981
1,30,2024,4141,0.173238
2,30,2025,4311,0.158863
3,30,2026,4097,0.159686
4,60,2023,474,0.168435
5,60,2024,457,0.172950
6,60,2025,477,0.163048
7,60,2026,453,0.157873
8,100,2023,33,0.144128
9,100,2024,19,0.167574


## The shape of the departure day
Scheduled origin-local hour is the exposure. The outcome remains arrival delay. Overnight sample sizes can be small.

In [6]:
con.execute('SELECT year,departure_hour,sum(scheduled_flights) AS scheduled_flights,sum(eligible_arrivals) AS eligible_arrivals,sum(delayed_flights) AS delayed_flights,sum(delayed_flights)*1.0/sum(eligible_arrivals) AS delay_rate FROM hourly WHERE month<=6 GROUP BY 1,2 ORDER BY 1,2').fetchdf()

,year,departure_hour,scheduled_flights,eligible_arrivals,delayed_flights,delay_rate
0,2023,0,5226.0,5129.0,1130.0,0.220316
1,2023,1,1841.0,1803.0,510.0,0.282862
2,2023,2,851.0,834.0,202.0,0.242206
3,2023,3,241.0,236.0,81.0,0.343220
4,2023,4,172.0,166.0,34.0,0.204819
...,...,...,...,...,...,...
91,2026,19,193540.0,187562.0,58362.0,0.311161
92,2026,20,156273.0,151781.0,48155.0,0.317266
93,2026,21,116992.0,113036.0,33210.0,0.293800
94,2026,22,78853.0,76221.0,20865.0,0.273743


## Airlines versus other carriers on the same routes
Only route-month cells with at least two carriers and at least 100 eligible arrivals per carrier enter. Competitor rates exclude the focal airline. The gap is weighted by the focal airline’s eligible arrivals. Route mix beyond this restricted sample and other confounding remain.

In [7]:
con.execute('SELECT year,carrier_name,sum(n) AS eligible_arrivals,count(*) AS matched_cells,sum(n*gap)/sum(n) AS difference_from_competitors FROM carrier_peers LEFT JOIN dim_airline USING(airline_key) WHERE month<=6 GROUP BY 1,2 ORDER BY 1,5').fetchdf()

,year,carrier_name,eligible_arrivals,matched_cells,difference_from_competitors
0,2023,Envoy Air,454.0,4,-0.166049
1,2023,Republic Airways,22519.0,115,-0.082488
2,2023,PSA Airlines,1747.0,12,-0.067666
3,2023,Endeavor Air,6003.0,46,-0.061189
4,2023,SkyWest Airlines,30605.0,162,-0.052740
5,2023,United Airlines,90318.0,530,-0.040000
6,2023,Delta Air Lines,108162.0,593,-0.037756
7,2023,Southwest Airlines,110877.0,612,0.014766
8,2023,American Airlines,88782.0,484,0.019532
9,2023,Alaska Airlines,34248.0,179,0.038280


## Unadjusted departure periods
Morning 06:00–10:59; evening 19:00–23:59. These periods are not equal in duration or traffic.

In [8]:
con.execute('SELECT year,departure_period,count(*) AS scheduled_flights,sum(eligible_arrival) AS eligible_arrivals,sum(delayed_flight)*1.0/sum(eligible_arrival) AS delay_rate FROM fact_flights WHERE month<=6 GROUP BY 1,2 ORDER BY 1,2').fetchdf()

,year,departure_period,scheduled_flights,eligible_arrivals,delay_rate
0,2023,Afternoon,786525,770245.0,0.283129
1,2023,Evening,576796,563493.0,0.305065
2,2023,Midday,791563,778926.0,0.220663
3,2023,Morning,1083247,1066669.0,0.145732
4,2023,Overnight,102438,100626.0,0.106195
5,2024,Afternoon,832078,816573.0,0.285857
6,2024,Evening,577424,566013.0,0.300795
7,2024,Midday,817817,805017.0,0.218323
8,2024,Morning,1125419,1109010.0,0.142355
9,2024,Overnight,108581,106852.0,0.106156


In [9]:
con.close()